# Notebook 3 — Teste de Resiliência

Verifica se o cluster HDFS + Spark continua operando após a queda de um DataNode.

**Cenário testado:**
1. Job Spark longo em execução
2. `docker stop datanode-2` no meio do processamento
3. Verificar se o job completa ou falha
4. Verificar integridade dos dados no HDFS (replicação = 2)

> **Pré-requisito:** Execute este notebook com 2 DataNodes ativos.

In [ ]:
import time
import json
import subprocess
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

def hdfs_report():
    r = subprocess.run('hdfs dfsadmin -report 2>&1 | head -30',
                       shell=True, capture_output=True, text=True)
    return r.stdout

def live_nodes():
    r = subprocess.run(
        'hdfs dfsadmin -report 2>&1 | grep "Live datanodes"',
        shell=True, capture_output=True, text=True
    )
    return r.stdout.strip()

print('Estado inicial do cluster:')
print(live_nodes())

In [ ]:
spark = SparkSession.builder \
    .appName('Teste_Resiliencia') \
    .master('spark://namenode:7077') \
    .config('spark.executor.memory', '1g') \
    .config('spark.task.maxFailures', '4') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')

df = spark.read.parquet('hdfs://namenode:9000/data/parquet/vendas.parquet')
print(f'Dataset carregado: {df.count():,} registros')

In [ ]:
# ============================================================
# INSTRUÇÕES:
# Após iniciar esta célula, abra outro terminal e execute:
#   docker stop datanode-2
# Observe se o job Spark completa mesmo assim.
# ============================================================

log = []

def checkpoint(msg):
    ts = time.strftime('%H:%M:%S')
    entry = f'[{ts}] {msg}'
    log.append(entry)
    print(entry)

checkpoint(f'Iniciando processamento | {live_nodes()}')
t0 = time.time()

try:
    # Job 1: agregação pesada com múltiplas passagens
    checkpoint('Job 1: Agregação por categoria + região (3 colunas derivadas)...')
    resultado1 = df.withColumn('ano', F.year('data_venda')) \
                   .groupBy('categoria', 'regiao', 'ano') \
                   .agg(
                       F.count('*').alias('n'),
                       F.sum('total').alias('receita'),
                       F.avg('desconto').alias('desconto_medio')
                   ) \
                   .orderBy('receita') \
                   .collect()
    checkpoint(f'Job 1 concluído | {len(resultado1)} grupos | {live_nodes()}')

    # Job 2: window function
    checkpoint('Job 2: Ranking de produtos por receita (window function)...')
    from pyspark.sql.window import Window
    w = Window.partitionBy('categoria').orderBy(F.desc('total'))
    resultado2 = df.withColumn('rank', F.rank().over(w)) \
                   .filter(F.col('rank') <= 10) \
                   .collect()
    checkpoint(f'Job 2 concluído | {len(resultado2)} registros | {live_nodes()}')

    # Job 3: join consigo mesmo simulando dimensões
    checkpoint('Job 3: Self-join para análise de co-ocorrência...')
    df_cat = df.groupBy('categoria').agg(F.sum('total').alias('total_categoria'))
    resultado3 = df.join(df_cat, on='categoria') \
                   .withColumn('pct_contribuicao',
                               F.col('total') / F.col('total_categoria') * 100) \
                   .agg(F.avg('pct_contribuicao').alias('media_pct')) \
                   .collect()
    checkpoint(f'Job 3 concluído | {resultado3} | {live_nodes()}')

    duracao = round(time.time() - t0, 2)
    checkpoint(f'SUCESSO — todos os jobs completaram em {duracao}s')
    status = 'SUCESSO'

except Exception as e:
    duracao = round(time.time() - t0, 2)
    checkpoint(f'FALHA após {duracao}s — Erro: {e}')
    status = 'FALHA'

print(f'\nStatus final: {status}')

In [ ]:
# Verificar integridade dos dados após a queda
print('=== Integridade dos dados no HDFS ===')
r = subprocess.run('hdfs fsck /data -files -blocks -locations 2>&1 | tail -20',
                   shell=True, capture_output=True, text=True)
print(r.stdout)

print('\n=== Estado atual do cluster ===')
print(hdfs_report())

In [ ]:
# Salvar relatório
relatorio = {
    'status': status,
    'duracao_segundos': duracao,
    'log': log,
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
}

with open('/data/resilience_report.json', 'w') as f:
    json.dump(relatorio, f, indent=2, ensure_ascii=False)

print('Relatório salvo em /data/resilience_report.json')
print(json.dumps(relatorio, indent=2, ensure_ascii=False))

spark.stop()